# Aeropulse — Silver Helpers

**Purpose:** Shared, dataset-agnostic transformation and load functions reused across the airport, carrier and flight silver notebooks — keeping dataset-specific decisions (rename maps, cast maps, key columns, DQ conditions) in each dataset's own notebook while the mechanics stay generic.

**Used by:** `airport-bronze-to-silver`, `carrier-bronze-to-silver`, `flight-bronze-to-silver` (via `%run silver-helper`)


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
# remove duplicates

def remove_duplicates(df, columns):
    return df.dropDuplicates(columns)


# remove nulls
def remove_nulls(df, columns):
    for c in columns:
        df = df.filter(F.col(c).isNotNull())
    return df

# rename column headers
def rename_column(df, rename_mapping):
    for old_name, new_name in rename_mapping.items():
        df = df.withColumnRenamed(old_name, new_name)
    return df

# trim whitespace
def trim_whitespaces(df):
    for c in df.columns:
        if df.schema[c].dataType == StringType():
            df = df.withColumn(c, F.trim(F.col(c)))
    return df

# cast columns to their right data type:
def cast_columns(df, cast_map):
    for column, target_type in cast_map.items():
        df = df.withColumn(column, F.col(column).cast(target_type))
    return df

# generate surrogate key
def add_sk_key(df, key_columns, sk_column_name = "sk"):
    return df.withColumn(
        sk_column_name,
        F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_columns]), 256)
    )

# add a boolean data quality flag
def add_dq_flag(df, condition, flag_column_name):
    return df.withColumn(flag_column_name, condition)

# write to silver function


In [ ]:
def write_to_silver(
    input_df,
    target_table,
    merge_condition,
    columns_to_update
):
    """
    Creates the schema if needed, creates the Delta table if it doesn't exist,
    otherwise merges the input dataframe into the target table.
    """
    if "." in target_table:
        schema_name = target_table.split(".")[0]
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

    final_df = (input_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp()))

    if not spark.catalog.tableExists(target_table):
        final_df.write.format('delta').mode('overwrite').saveAsTable(target_table)
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {column: f"s.{column}" for column in columns_to_update if column != "created_timestamp"}
        update_map["updated_timestamp"] = "s.updated_timestamp"

        (delta_table.alias("t")
            .merge(final_df.alias("s"), merge_condition)
            .whenMatchedUpdate(
                condition="s.batch_id >= t.batch_id",
                set=update_map
            )
            .whenNotMatchedInsertAll()
            .execute())